<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/05_regression_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Statistical analysis (SOURCE-FIXED pipeline)

This notebook MUST be run only after the source-leakage-fixed classifier notebook (03) and the newly rerun Grad-CAM notebook (04).
It includes strict freshness/synchronization checks so it will STOP rather than analyze stale CSVs.


## Setup — load ResNet18 regression-ready data, with a freshness check

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.robust.norms as robust_norms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import mannwhitneyu

# ============================================================
# EXACT OUTPUT FILES WRITTEN BY NOTEBOOK 04
# ============================================================
RESNET_CSV = "/content/drive/MyDrive/CASIA2.0/regression_ready_resnet.csv"
EFFNET_CSV = "/content/drive/MyDrive/CASIA2.0/regression_ready_effnet.csv"

assert os.path.exists(RESNET_CSV), f"Missing file: {RESNET_CSV}"
assert os.path.exists(EFFNET_CSV), f"Missing file: {EFFNET_CSV}"

resnet_df = pd.read_csv(RESNET_CSV)
effnet_df = pd.read_csv(EFFNET_CSV)

# ============================================================
# STRICT SYNCHRONIZATION CHECK AGAINST THE CURRENT SOURCE-FIXED
# NOTEBOOK 04 OUTPUTS.
#
# Current notebook 04 printed:
#   ResNet18 mean IoU       = 0.110172
#   ResNet18 mean AUC-IoU   = 0.075967
#   EfficientNet-B0 mean IoU     = 0.086194
#   EfficientNet-B0 mean AUC-IoU = 0.055325
#
# If these assertions fail, STOP: notebook 05 is reading stale
# or different CSVs. Re-run notebook 04 through its save/merge cells.
# ============================================================
EXPECTED_N = 1828
EXPECTED_RESNET_IOU = 0.110172
EXPECTED_RESNET_AUC = 0.075967
EXPECTED_EFFNET_IOU = 0.086194
EXPECTED_EFFNET_AUC = 0.055325

assert len(resnet_df) == EXPECTED_N, (
    f"ResNet CSV has {len(resnet_df)} rows, expected {EXPECTED_N}."
)
assert len(effnet_df) == EXPECTED_N, (
    f"EfficientNet CSV has {len(effnet_df)} rows, expected {EXPECTED_N}."
)

assert resnet_df["filename"].nunique() == EXPECTED_N, "Duplicate/missing ResNet filenames."
assert effnet_df["filename"].nunique() == EXPECTED_N, "Duplicate/missing EfficientNet filenames."

def close_enough(actual, expected, tol=5e-5):
    return abs(float(actual) - float(expected)) <= tol

r_iou = resnet_df["iou"].mean()
r_auc = resnet_df["auc_iou"].mean()
e_iou = effnet_df["iou"].mean()
e_auc = effnet_df["auc_iou"].mean()

assert close_enough(r_iou, EXPECTED_RESNET_IOU), (
    f"STALE/WRONG ResNet CSV: mean IoU={r_iou:.6f}, "
    f"but notebook 04 produced {EXPECTED_RESNET_IOU:.6f}."
)
assert close_enough(r_auc, EXPECTED_RESNET_AUC), (
    f"STALE/WRONG ResNet CSV: mean AUC-IoU={r_auc:.6f}, "
    f"but notebook 04 produced {EXPECTED_RESNET_AUC:.6f}."
)
assert close_enough(e_iou, EXPECTED_EFFNET_IOU), (
    f"STALE/WRONG EfficientNet CSV: mean IoU={e_iou:.6f}, "
    f"but notebook 04 produced {EXPECTED_EFFNET_IOU:.6f}."
)
assert close_enough(e_auc, EXPECTED_EFFNET_AUC), (
    f"STALE/WRONG EfficientNet CSV: mean AUC-IoU={e_auc:.6f}, "
    f"but notebook 04 produced {EXPECTED_EFFNET_AUC:.6f}."
)

# ============================================================
# VERIFY BOTH ARCHITECTURES CONTAIN EXACTLY THE SAME 1,828
# EVALUATION IMAGES AND THE SAME IMAGE-LEVEL FEATURES.
# ============================================================
required = {
    "filename", "iou", "auc_iou", "polarity",
    "splice_size_frac", "abs_contrast"
}
for name, df in [("ResNet", resnet_df), ("EfficientNet", effnet_df)]:
    missing = required - set(df.columns)
    assert not missing, f"{name} CSV missing columns: {sorted(missing)}"

r_names = set(resnet_df["filename"].astype(str))
e_names = set(effnet_df["filename"].astype(str))
assert r_names == e_names, (
    f"Architecture CSVs do not contain identical images. "
    f"Only ResNet={len(r_names-e_names)}, only EfficientNet={len(e_names-r_names)}"
)

pair_check = resnet_df[
    ["filename", "polarity", "splice_size_frac", "abs_contrast"]
].merge(
    effnet_df[
        ["filename", "polarity", "splice_size_frac", "abs_contrast"]
    ],
    on="filename",
    suffixes=("_r", "_e"),
    validate="one_to_one"
)

assert (pair_check["polarity_r"] == pair_check["polarity_e"]).all(), \
    "Polarity differs for the same image across architecture CSVs."
assert np.allclose(
    pair_check["splice_size_frac_r"],
    pair_check["splice_size_frac_e"],
    equal_nan=True
), "splice_size_frac differs across architecture CSVs."
assert np.allclose(
    pair_check["abs_contrast_r"],
    pair_check["abs_contrast_e"],
    equal_nan=True
), "abs_contrast differs across architecture CSVs."

# Encode polarity identically
resnet_df["polarity_bin"] = (resnet_df["polarity"] == "dark_on_bright").astype(int)
effnet_df["polarity_bin"] = (effnet_df["polarity"] == "dark_on_bright").astype(int)

# Use ONE common median because size is an image property, not an architecture property
common_median_size = resnet_df["splice_size_frac"].median()
resnet_df["large_splice_median"] = (
    resnet_df["splice_size_frac"] >= common_median_size
).astype(int)
effnet_df["large_splice_median"] = (
    effnet_df["splice_size_frac"] >= common_median_size
).astype(int)

print("✓ SOURCE-FIXED NOTEBOOK 04 OUTPUTS VERIFIED")
print(f"ResNet CSV modified:       {time.ctime(os.path.getmtime(RESNET_CSV))}")
print(f"EfficientNet CSV modified: {time.ctime(os.path.getmtime(EFFNET_CSV))}")
print(f"n per architecture: {EXPECTED_N}")
print(f"ResNet mean IoU / AUC-IoU:       {r_iou:.6f} / {r_auc:.6f}")
print(f"EfficientNet mean IoU / AUC-IoU: {e_iou:.6f} / {e_auc:.6f}")
print(f"Common median splice size: {common_median_size:.6f}")
print("✓ Same 1,828 filenames and image-level predictors across architectures")


Mounted at /content/drive
✓ SOURCE-FIXED NOTEBOOK 04 OUTPUTS VERIFIED
ResNet CSV modified:       Fri Aug 28 14:53:39 2026
EfficientNet CSV modified: Fri Aug 28 14:53:39 2026
n per architecture: 1828
ResNet mean IoU / AUC-IoU:       0.110172 / 0.075967
EfficientNet mean IoU / AUC-IoU: 0.086194 / 0.055325
Common median splice size: 0.062140
✓ Same 1,828 filenames and image-level predictors across architectures


In [2]:
# ============================================================
# DIRECT CROSS-ARCHITECTURE TEST
# architecture × size × polarity
# Same 1,828 images are evaluated by both architectures, so
# standard errors are clustered by image_id.
# ============================================================

res = resnet_df[
    ["filename", "iou", "polarity_bin",
     "large_splice_median", "abs_contrast"]
].copy()
eff = effnet_df[
    ["filename", "iou", "polarity_bin",
     "large_splice_median", "abs_contrast"]
].copy()

res["architecture"] = "resnet18"
eff["architecture"] = "efficientnet"

res = res.rename(columns={"filename": "image_id"})
eff = eff.rename(columns={"filename": "image_id"})

combined_df = pd.concat([res, eff], ignore_index=True)
combined_df["arch_bin"] = (
    combined_df["architecture"] == "efficientnet"
).astype(int)

analysis_cols = [
    "image_id", "iou", "polarity_bin",
    "large_splice_median", "abs_contrast", "arch_bin"
]
combined_df = combined_df.dropna(subset=analysis_cols).copy()

assert combined_df["image_id"].nunique() == 1828
assert len(combined_df) == 3656

formula_3way = (
    "iou ~ polarity_bin * large_splice_median * arch_bin + abs_contrast"
)

ols_3way = smf.ols(
    formula=formula_3way,
    data=combined_df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": combined_df["image_id"]}
)

term3 = "polarity_bin:large_splice_median:arch_bin"

print(ols_3way.summary())
print("\n--- THREE-WAY INTERACTION ---")
print(f"Coefficient: {ols_3way.params[term3]:.6f}")
print(f"SE:          {ols_3way.bse[term3]:.6f}")
print(f"p-value:     {ols_3way.pvalues[term3]:.6g}")
print(
    "95% CI:     ",
    tuple(ols_3way.conf_int().loc[term3].round(6))
)

beta_resnet_3way = ols_3way.params[
    "polarity_bin:large_splice_median"
]
beta_effnet_3way = (
    beta_resnet_3way + ols_3way.params[term3]
)

print("\n--- ARCHITECTURE-SPECIFIC SIZE × POLARITY EFFECTS ---")
print(f"ResNet18:      {beta_resnet_3way:+.6f}")
print(f"EfficientNet:  {beta_effnet_3way:+.6f}")


                            OLS Regression Results                            
Dep. Variable:                    iou   R-squared:                       0.292
Model:                            OLS   Adj. R-squared:                  0.290
Method:                 Least Squares   F-statistic:                     202.7
Date:                Fri, 28 Aug 2026   Prob (F-statistic):          1.27e-245
Time:                        15:21:10   Log-Likelihood:                 2869.0
No. Observations:                3656   AIC:                            -5720.
Df Residuals:                    3647   BIC:                            -5664.
Df Model:                           8                                         
Covariance Type:              cluster                                         
                                                coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------

## Step 1 — Continuous specifications (tested first, in this order)

In [3]:
# --- Linear ---
model_linear = "iou ~ polarity_bin * splice_size_frac + abs_contrast"
ols_linear = smf.ols(formula=model_linear, data=resnet_df).fit(cov_type="HC3")
print("=== Linear (HC3) ===")
print(ols_linear.summary().tables[1])

# --- Log-transformed ---
resnet_df["log_splice_size"] = np.log(resnet_df["splice_size_frac"] + 0.001)
model_log = "iou ~ polarity_bin * log_splice_size + abs_contrast"
ols_log = smf.ols(formula=model_log, data=resnet_df).fit(cov_type="HC3")
print("\n=== Log-transformed (HC3) ===")
print(ols_log.summary().tables[1])

# --- Quadratic ---
resnet_df["splice_size_sq"] = resnet_df["splice_size_frac"] ** 2
model_quad = "iou ~ polarity_bin * splice_size_frac + polarity_bin * splice_size_sq + abs_contrast"
ols_quad = smf.ols(formula=model_quad, data=resnet_df).fit(cov_type="HC3")
print("\n=== Quadratic (HC3) ===")
print(ols_quad.summary().tables[1])

=== Linear (HC3) ===
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                         0.0598      0.005     13.279      0.000       0.051       0.069
polarity_bin                     -0.0069      0.005     -1.320      0.187      -0.017       0.003
splice_size_frac                  0.3854      0.022     17.167      0.000       0.341       0.429
polarity_bin:splice_size_frac     0.0742      0.056      1.321      0.187      -0.036       0.184
abs_contrast                  -2.319e-05    7.5e-05     -0.309      0.757      -0.000       0.000

=== Log-transformed (HC3) ===
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                        0.2632      0.010     26.413      0

## Step 2 — Diagnose the linear specification's instability

In [4]:
influence = ols_linear.get_influence()
cooks_d = influence.cooks_distance[0]
hat_values = influence.hat_matrix_diag

threshold = 4 / len(resnet_df)
n_influential = (cooks_d > threshold).sum()
print(f"Influential points (Cook's D > 4/n): {n_influential} of {len(resnet_df)}")

resnet_df["leverage"] = hat_values
resnet_df["dist_from_mean_size"] = np.abs(resnet_df["splice_size_frac"] - resnet_df["splice_size_frac"].mean())
corr_leverage = resnet_df["leverage"].corr(resnet_df["dist_from_mean_size"])
print(f"Correlation between leverage and distance from mean splice size: {corr_leverage:.3f}")

influential_df = resnet_df[cooks_d > threshold]
normal_df = resnet_df[cooks_d <= threshold]
print(f"\nInfluential points — mean splice_size_frac: {influential_df['splice_size_frac'].mean():.3f}")
print(f"Non-influential points — mean splice_size_frac: {normal_df['splice_size_frac'].mean():.3f}")

Influential points (Cook's D > 4/n): 126 of 1828
Correlation between leverage and distance from mean splice size: 0.752

Influential points — mean splice_size_frac: 0.423
Non-influential points — mean splice_size_frac: 0.112


## Step 3 — Robust regression on continuous splice size (Huber / Tukey biweight M-estimation)
Tests whether down-weighting — rather than binning — the high-leverage large-splice points recovers a stable continuous interaction.

**Note:** `statsmodels.RLM` implements M-estimation (IRLS from the OLS start), not a true MM-estimator (Yohai, 1987), which would require a high-breakdown initial fit not available in a standard Python stack. Report this as M-estimation, not MM-estimation.

In [5]:
resnet_df["interaction"] = resnet_df["polarity_bin"] * resnet_df["splice_size_frac"]
X_robust = resnet_df[["polarity_bin", "splice_size_frac", "interaction", "abs_contrast"]].copy()
X_robust = sm.add_constant(X_robust)
y_robust = resnet_df["iou"]

huber_fit = sm.RLM(y_robust, X_robust, M=robust_norms.HuberT()).fit()
print("=== Robust regression: Huber's T M-estimator ===")
print(huber_fit.summary())

tukey_fit = sm.RLM(y_robust, X_robust, M=robust_norms.TukeyBiweight()).fit()
print("\n=== Robust regression: Tukey biweight M-estimator ===")
print(tukey_fit.summary())

resnet_df["huber_weight"] = huber_fit.weights
resnet_df["tukey_weight"] = tukey_fit.weights
influential_mask = cooks_d > threshold
print("\n--- Mean estimator weight: high-Cook's-D points vs. rest ---")
print(f"Huber  - influential: {resnet_df.loc[influential_mask, 'huber_weight'].mean():.3f}  "
      f"| rest: {resnet_df.loc[~influential_mask, 'huber_weight'].mean():.3f}")
print(f"Tukey  - influential: {resnet_df.loc[influential_mask, 'tukey_weight'].mean():.3f}  "
      f"| rest: {resnet_df.loc[~influential_mask, 'tukey_weight'].mean():.3f}")

print("\n--- Interaction term ---")
print(f"Huber:  coef={huber_fit.params['interaction']:.4f}, p={huber_fit.pvalues['interaction']:.4f}")
print(f"Tukey:  coef={tukey_fit.params['interaction']:.4f}, p={tukey_fit.pvalues['interaction']:.4f}")

=== Robust regression: Huber's T M-estimator ===
                    Robust linear Model Regression Results                    
Dep. Variable:                    iou   No. Observations:                 1828
Model:                            RLM   Df Residuals:                     1823
Method:                          IRLS   Df Model:                            4
Norm:                          HuberT                                         
Scale Est.:                       mad                                         
Cov Type:                          H1                                         
Date:                Fri, 28 Aug 2026                                         
Time:                        15:21:35                                         
No. Iterations:                    33                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

## Step 4 — Threshold-based specification (primary result)
Splice size dichotomized at two cutoffs — sample median and top-33% quantile.

In [6]:
median_size = resnet_df["splice_size_frac"].median()
resnet_df["large_splice_median"] = (resnet_df["splice_size_frac"] >= median_size).astype(int)
model_median = "iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_median = smf.ols(formula=model_median, data=resnet_df).fit(cov_type="HC3")
print(f"=== Median split (cutoff={median_size:.4f}) ===")
print(ols_median.summary().tables[1])

cutoff_33 = resnet_df["splice_size_frac"].quantile(0.67)
resnet_df["large_splice_top33"] = (resnet_df["splice_size_frac"] >= cutoff_33).astype(int)
model_top33 = "iou ~ polarity_bin * large_splice_top33 + abs_contrast"
ols_top33 = smf.ols(formula=model_top33, data=resnet_df).fit(cov_type="HC3")
print(f"\n=== Top-33% split (cutoff={cutoff_33:.4f}) ===")
print(ols_top33.summary().tables[1])

=== Median split (cutoff=0.0621) ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.0415      0.004     10.673      0.000       0.034       0.049
polarity_bin                        -0.0025      0.003     -0.846      0.398      -0.008       0.003
large_splice_median                  0.1404      0.007     20.908      0.000       0.127       0.154
polarity_bin:large_splice_median    -0.0143      0.010     -1.440      0.150      -0.034       0.005
abs_contrast                      6.021e-05   7.33e-05      0.821      0.412   -8.35e-05       0.000

=== Top-33% split (cutoff=0.1211) ===
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept        

## Step 5 — Distribution-free validation (Mann-Whitney U)
`stratified_mannwhitney` prints n, p, and r at each cutoff. Direction check printed explicitly — states which polarity group has the higher mean at each cutoff, so a sign flip (like the one caught in the previous run) is visible immediately rather than requiring separate raw-means inspection.

In [7]:
def stratified_mannwhitney(df, outcome="iou"):
    for pct in [0.40, 0.33, 0.30, 0.25, 0.20]:
        cutoff = df["splice_size_frac"].quantile(1 - pct)
        large = df[df["splice_size_frac"] >= cutoff]
        dark = large[large["polarity_bin"] == 1][outcome]
        bright = large[large["polarity_bin"] == 0][outcome]
        stat, p = mannwhitneyu(dark, bright, alternative="two-sided")
        r = 1 - (2 * stat) / (len(dark) * len(bright))
        higher = "dark_on_bright" if dark.mean() > bright.mean() else "bright_on_dark"
        print(f"Top {int(pct*100)}%: n_dark={len(dark)}, n_bright={len(bright)}, p={p:.4f}, r={r:.3f} "
              f"| mean(dark)={dark.mean():.4f}, mean(bright)={bright.mean():.4f}, higher={higher}")

print("=== ResNet18, outcome=iou ===")
stratified_mannwhitney(resnet_df, outcome="iou")

=== ResNet18, outcome=iou ===
Top 40%: n_dark=278, n_bright=453, p=0.3684, r=0.040 | mean(dark)=0.1901, mean(bright)=0.1983, higher=bright_on_dark
Top 33%: n_dark=214, n_bright=389, p=0.4947, r=0.034 | mean(dark)=0.2030, mean(bright)=0.2080, higher=bright_on_dark
Top 30%: n_dark=181, n_bright=368, p=0.9619, r=0.003 | mean(dark)=0.2116, mean(bright)=0.2094, higher=dark_on_bright
Top 25%: n_dark=136, n_bright=321, p=0.8186, r=0.014 | mean(dark)=0.2206, mean(bright)=0.2177, higher=dark_on_bright
Top 20%: n_dark=99, n_bright=267, p=0.6460, r=0.031 | mean(dark)=0.2269, mean(bright)=0.2278, higher=bright_on_dark


## Step 6 — Raw group means (top-33% cutoff)
Recomputed directly from `resnet_df` in this cell — not read from any earlier cached output — to avoid the staleness issue from the previous run.

In [8]:
large = resnet_df[resnet_df["large_splice_top33"] == 1]
means = large.groupby("polarity_bin")["iou"].agg(["mean", "std", "count"])
print(means)
print("\n(polarity_bin: 0 = bright_on_dark, 1 = dark_on_bright)")
higher = "dark_on_bright (1)" if means.loc[1, "mean"] > means.loc[0, "mean"] else "bright_on_dark (0)"
print(f"Higher mean IoU in this stratum: {higher}")

                  mean       std  count
polarity_bin                           
0             0.208040  0.154354    389
1             0.202968  0.162667    214

(polarity_bin: 0 = bright_on_dark, 1 = dark_on_bright)
Higher mean IoU in this stratum: bright_on_dark (0)


## Step 7 — Robustness check: AUC-IoU (threshold-free outcome)
Repeats the median-split regression and stratified Mann-Whitney using `auc_iou` instead of `iou`.

In [9]:
model_median_auc = "auc_iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_median_auc = smf.ols(formula=model_median_auc, data=resnet_df).fit(cov_type="HC3")
print("=== ResNet18, median split, outcome=auc_iou ===")
print(ols_median_auc.summary().tables[1])

print("\n=== ResNet18, stratified Mann-Whitney, outcome=auc_iou ===")
stratified_mannwhitney(resnet_df, outcome="auc_iou")

=== ResNet18, median split, outcome=auc_iou ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.0459      0.004     12.906      0.000       0.039       0.053
polarity_bin                        -0.0036      0.004     -0.984      0.325      -0.011       0.004
large_splice_median                  0.0636      0.005     13.919      0.000       0.055       0.073
polarity_bin:large_splice_median     0.0058      0.007      0.812      0.417      -0.008       0.020
abs_contrast                     -2.635e-05   4.87e-05     -0.541      0.588      -0.000     6.9e-05

=== ResNet18, stratified Mann-Whitney, outcome=auc_iou ===
Top 40%: n_dark=278, n_bright=453, p=0.9400, r=0.003 | mean(dark)=0.1202, mean(bright)=0.1127, higher=dark_on_bright
Top 33%: n_dark=214, n_bright=389, p=0.8460, r=0.010 | mean(dark)=0.

## Step 8 — Cross-architecture replication (EfficientNet-B0)
Loaded fresh from disk, same as ResNet18 above — not reused from any earlier session variable.

In [10]:
# IMPORTANT: effnet_df was already loaded and VERIFIED in the setup cell.
# Do NOT reload it here; this prevents accidental analysis of a different/stale file.

print(f"EfficientNet verified n = {len(effnet_df)}, mean IoU = {effnet_df['iou'].mean():.6f}")

# Reuse the exact same common median cutoff used for ResNet
effnet_df["large_splice_median"] = (
    effnet_df["splice_size_frac"] >= common_median_size
).astype(int)

model_median_eff = "iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_median_eff = smf.ols(
    formula=model_median_eff,
    data=effnet_df
).fit(cov_type="HC3")

print("\n=== EfficientNet-B0, median split, outcome=iou ===")
print(ols_median_eff.summary().tables[1])

model_median_eff_auc = "auc_iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_median_eff_auc = smf.ols(
    formula=model_median_eff_auc,
    data=effnet_df
).fit(cov_type="HC3")

print("\n=== EfficientNet-B0, median split, outcome=auc_iou ===")
print(ols_median_eff_auc.summary().tables[1])

print("\n=== EfficientNet-B0, stratified Mann-Whitney, outcome=iou ===")
stratified_mannwhitney(effnet_df, outcome="iou")

print("\n=== EfficientNet-B0, stratified Mann-Whitney, outcome=auc_iou ===")
stratified_mannwhitney(effnet_df, outcome="auc_iou")


EfficientNet verified n = 1828, mean IoU = 0.086194

=== EfficientNet-B0, median split, outcome=iou ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.0160      0.004      4.457      0.000       0.009       0.023
polarity_bin                        -0.0028      0.002     -1.474      0.141      -0.006       0.001
large_splice_median                  0.1632      0.008     21.327      0.000       0.148       0.178
polarity_bin:large_splice_median    -0.0663      0.011     -6.210      0.000      -0.087      -0.045
abs_contrast                      8.852e-05   7.61e-05      1.163      0.245   -6.06e-05       0.000

=== EfficientNet-B0, median split, outcome=auc_iou ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------

In [11]:
# ============================================================
# ROBUSTNESS TEST:
# Three-way Architecture × Size × Polarity interaction
# Outcome = AUC-IoU instead of mean-threshold IoU
# ============================================================

# Prepare ResNet
res_auc = resnet_df[
    [
        "filename",
        "auc_iou",
        "polarity_bin",
        "large_splice_median",
        "abs_contrast"
    ]
].copy()

res_auc["architecture"] = "resnet18"
res_auc["image_id"] = res_auc["filename"].astype(str)


# Prepare EfficientNet
eff_auc = effnet_df[
    [
        "filename",
        "auc_iou",
        "polarity_bin",
        "large_splice_median",
        "abs_contrast"
    ]
].copy()

eff_auc["architecture"] = "efficientnet"
eff_auc["image_id"] = eff_auc["filename"].astype(str)


# Combine
combined_auc = pd.concat(
    [res_auc, eff_auc],
    ignore_index=True
)

combined_auc["arch_bin"] = (
    combined_auc["architecture"] == "efficientnet"
).astype(int)


# Remove missing values
combined_auc = combined_auc.dropna(
    subset=[
        "image_id",
        "auc_iou",
        "polarity_bin",
        "large_splice_median",
        "abs_contrast",
        "arch_bin"
    ]
).copy()


# Sanity checks
print("Total observations:", len(combined_auc))
print("Unique images:", combined_auc["image_id"].nunique())
print(combined_auc["architecture"].value_counts())

assert combined_auc["image_id"].nunique() == 1828
assert len(combined_auc) == 3656


# ============================================================
# THREE-WAY MODEL
# Same model as before, but outcome = auc_iou
#
# Cluster by image because the SAME images are evaluated
# by ResNet18 and EfficientNet-B0.
# ============================================================

formula_auc_3way = """
auc_iou ~ polarity_bin
          * large_splice_median
          * arch_bin
          + abs_contrast
"""

fit_auc_3way = smf.ols(
    formula=formula_auc_3way,
    data=combined_auc
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": combined_auc["image_id"]
    }
)


# Full table
print("\n=== AUC-IoU THREE-WAY MODEL ===")
print(fit_auc_3way.summary().tables[1])


# ============================================================
# KEY RESULT
# ============================================================

term3 = "polarity_bin:large_splice_median:arch_bin"

print("\n=== AUC-IoU THREE-WAY INTERACTION ===")

print(
    f"Coefficient: "
    f"{fit_auc_3way.params[term3]:.6f}"
)

print(
    f"SE:          "
    f"{fit_auc_3way.bse[term3]:.6f}"
)

print(
    f"p-value:     "
    f"{fit_auc_3way.pvalues[term3]:.6g}"
)

print(
    "95% CI:     ",
    tuple(
        fit_auc_3way
        .conf_int()
        .loc[term3]
        .round(6)
    )
)


# ============================================================
# Architecture-specific Size × Polarity interactions
# ============================================================

beta_resnet_auc = fit_auc_3way.params[
    "polarity_bin:large_splice_median"
]

beta_effnet_auc = (
    beta_resnet_auc
    +
    fit_auc_3way.params[
        "polarity_bin:large_splice_median:arch_bin"
    ]
)

print(
    "\n=== ARCHITECTURE-SPECIFIC "
    "SIZE × POLARITY EFFECTS (AUC-IoU) ==="
)

print(
    f"ResNet18:      {beta_resnet_auc:+.6f}"
)

print(
    f"EfficientNet:  {beta_effnet_auc:+.6f}"
)

Total observations: 3656
Unique images: 1828
architecture
resnet18        1828
efficientnet    1828
Name: count, dtype: int64

=== AUC-IoU THREE-WAY MODEL ===
                                                coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------
Intercept                                     0.0439      0.003     13.622      0.000       0.038       0.050
polarity_bin                                 -0.0038      0.004     -1.046      0.295      -0.011       0.003
large_splice_median                           0.0634      0.005     13.915      0.000       0.054       0.072
polarity_bin:large_splice_median              0.0066      0.007      0.929      0.353      -0.007       0.021
arch_bin                                     -0.0285      0.003     -9.903      0.000      -0.034      -0.023
polarity_bin:arch_bin                         0.0023      0.004      0.

## Summary
Fill in after running: populate coefficients/p-values/direction from the actual output above before writing the paper's Results section. Pay attention to the `higher=` field printed in Step 5/6 — confirm which polarity group actually has the higher IoU on this run before reusing any wording from a previous draft, since direction is not guaranteed to match earlier (leaked) results.